`conda activate r_python`

# import the functions to plot the respective functions
source('scripts/plotSingleTaskRNA.R');

df <- read.table('data/metrics_multi_integration_dmg_atlas.csv', 
                 sep=',', header=T) %>%
                `row.names<-`(c('MultiVIModality_hvg_embed',
                                'MultiVISampleID_hvg_embed',
                                'GlueModality_hvg_embed',
                                'GlueModalityBatch_hvg_embed',
                                'MultigrateModality_hvg_embed',
                                'MultigrateModalityBatch_hvg_embed')) %>%
                select(-X)
df

write.table(df, 'data/metrics_multi_integration_dmg_atlas_modified.csv', row.names = TRUE,
           col.names = TRUE, quote = FALSE, sep = ',')

dim(df)

rownames(df) <- str_replace(rownames(df), "\\+", 'and')
rownames(df) <- str_replace(rownames(df), "\\+", 'and')
rownames(df) <- str_replace(rownames(df), "\\+", 'and')

df <- df[grepl('Roska', rownames(df)) == FALSE,]
df <- df[grepl('_E', rownames(df)),]
# df <- df[grepl('_L', rownames(df)),]

dim(df)

metrics_path_all = 'data/metrics_multi_integration_dmg_atlas_modified.csv'

csv_metrics_path <- metrics_path_all

outdir = 'figures'
print(outdir)
weight_batch = 0.4
metrics_tab_lab <- read.table(csv_metrics_path, sep = ",", header = TRUE) %>% 
                    tibble::rownames_to_column('X')

metrics_tab_lab

metrics_tab_lab$X <- str_replace(metrics_tab_lab$X, "\\+", 'and')
metrics_tab_lab$X <- str_replace(metrics_tab_lab$X, "\\+", 'and')
metrics_tab_lab$X <- str_replace(metrics_tab_lab$X, "\\+", 'and')

# print(metrics_tab_lab)
# get metrics names from columns
metrics <- colnames(metrics_tab_lab)[-1]
metrics <- gsub("\\.", "/", metrics)
metrics <- gsub("_", " ", metrics)
# print(metrics)
metrics <- plyr::mapvalues(metrics, from = c("ASW label", "ASW label/batch", "cell cycle conservation", "hvg overlap", "trajectory", "graph conn", "iLISI", "cLISI"), 
                         to = c("Cell type ASW", "Batch ASW", "CC conservation", "HVG conservation", "trajectory conservation", "graph connectivity", "graph iLISI", "graph cLISI"))
print(metrics)
# stopifnot(FALSE)

# metrics names as they are supposed to be ordered
group_batch <- c("PCR batch", "Batch ASW", "graph iLISI", "graph connectivity", "kBET")
group_bio <- c("NMI cluster/label", "ARI cluster/label", "Cell type ASW", 
             "isolated label F1", "isolated label silhouette", "graph cLISI", "CC conservation", "HVG conservation", "trajectory conservation")
# set original values of number of metrics
n_metrics_batch_original <- sum(group_batch %in% metrics)
n_metrics_bio_original <- sum(group_bio %in% metrics)

# order metrics present in the table
matching.order <- match(c(group_batch, group_bio), metrics)
metrics.ord <- metrics[matching.order[!is.na(matching.order)]]

# get methods info from rownames
methods_info_full  <- as.character(metrics_tab_lab[,1])

# in case methods names start with /
if(substring(methods_info_full[1], 1, 1) == "/"){
methods_info_full <- sub("/", "", methods_info_full)
}

# Remove trvae full
ind.trvae_full <- grep("trvae_full", methods_info_full)
if(length(ind.trvae_full) > 0){
methods_info_full <- methods_info_full[-ind.trvae_full]
metrics_tab_lab <- metrics_tab_lab[-ind.trvae_full,]
}

# data scenarios to be saved in file name
data.scenarios <- unique(unlist(sapply(str_split(methods_info_full, "/"), function(x) x[2])))

print(data.scenarios)


methods_info_full

###### Plot one figure for each data task

methods_info <- methods_info_full
metrics_tab_sub <- metrics_tab_lab

metrics_tab_sub

print(methods_info)    
# info on scaling data
scaling <- rep(F, length(methods_info)) # sapply(str_split(methods_info, "/"), function(x) x[3])


# info on HVG selection
hvg <- sapply(str_split(methods_info, "_"), function(x) x[2])
hvg <- plyr::mapvalues(hvg, from = c("hvg", "full_feature"), to = c("HVG", "FULL"))

print(hvg)

methods <- sapply(str_split(methods_info, "_"), function(x) x[1])

methods_name <- sapply(str_split(methods, "_"), function(x) x[1])
methods_name <- capitalize(methods_name)
methods_name <- plyr::mapvalues(methods_name, 
                                from = c("Multivi", "Scvi", "Seurat", "Seuratrpca", "Mnn", "Bbknn", "Trvae", "Scvi", "Liger", "Combat", "Saucie", "Fastmnn", "Desc", "Scanvi", "Scgen"), 
                                to = c("MultiVI", "scVI", "Seurat v3 CCA", "Seurat v3 RPCA", "MNN", "BBKNN", "trVAE", "scVI", "LIGER", "ComBat", "SAUCIE", "fastMNN", "DESC", "scANVI*", "scGen*"))


method_groups <- sapply(str_split(methods_info, "_"), function(x) x[3])
method_groups <- plyr::mapvalues(method_groups, 
                                  from = c("knn", "embed", "full"), 
                                  to = c("graph", "embed", "gene"))



##### Create dataframe 
metrics_tab <- as.data.frame(metrics_tab_sub[, -1])
metrics_tab[metrics_tab == ""] <- NA
colnames(metrics_tab) <- metrics

#add Methods column
metrics_tab <- add_column(metrics_tab, "Method" = methods_name, .before = 1)

# reorder columns by metrics
col.ordered <- c("Method", metrics.ord)
metrics_tab <- metrics_tab[, col.ordered]

## Remove columns that are full NAs
na.col <- apply(metrics_tab, 2, function(x) sum(is.na(x)) == nrow(metrics_tab))
# redefine numbers of metrics per group
print('metrics columns')
print(colnames(metrics_tab))
if(sum(colnames(metrics_tab)[na.col] %in%  group_batch) > 0){
  n_metrics_batch <- n_metrics_batch_original - sum(colnames(metrics_tab)[na.col] %in%  group_batch)
} else {
  n_metrics_batch <- n_metrics_batch_original
}

if(sum(colnames(metrics_tab)[na.col] %in% group_bio) > 0){
  n_metrics_bio <- n_metrics_bio_original - sum(colnames(metrics_tab)[na.col] %in% group_bio)
} else{
  n_metrics_bio <- n_metrics_bio_original
}
  
metrics_tab <- metrics_tab[, !na.col]

print(dim(metrics_tab))

## Scores should be already scaled [0,1] - however, we aim to compute the scores based on the min-max scaled metrics
print('here...')
print(rownames(metrics_tab))
print(dim(metrics_tab))
scaled_metrics_tab <- as.matrix(metrics_tab[, -1])
print(dim(scaled_metrics_tab))

use_scale_minmax <- F
if(use_scale_minmax)
   scaled_metrics_tab <- apply(scaled_metrics_tab, 2, function(x) scale_minmax(x))

# calculate average score by group and overall
print(scaled_metrics_tab)
print(n_metrics_batch)
print(scaled_metrics_tab[, 1:n_metrics_batch])
score_group_batch <- rowMeans(scaled_metrics_tab[, 1:n_metrics_batch], na.rm = T)
score_group_bio <- rowMeans(scaled_metrics_tab[, (1+n_metrics_batch):ncol(scaled_metrics_tab)], 
                          na.rm = T)

score_all <- (weight_batch*score_group_batch + (1-weight_batch)*score_group_bio)

metrics_tab <- add_column(metrics_tab, "Overall Score" = score_all, .after = "Method")
metrics_tab <- add_column(metrics_tab, "Batch Correction" = score_group_batch, .after = "Overall Score")
metrics_tab <- add_column(metrics_tab, "Bio conservation" = score_group_bio, .after = 3+n_metrics_batch)

metrics_tab <- add_column(metrics_tab, "Output" = method_groups, .after = "Method")
metrics_tab <- add_column(metrics_tab, "Features" = hvg, .after = "Output")
metrics_tab <- add_column(metrics_tab, "Scaling" = scaling, .after = "Features")

# order methods by the overall score
metrics_tab <- metrics_tab[order(metrics_tab$`Overall Score`,  decreasing = T), ]
write.csv(metrics_tab, file = paste0(outdir, "/", "multi_integration_dmg_atlas_summary_scores.csv"), quote = F)


# Delete rows that are empty
rowsNA <- which(is.na(metrics_tab$`Overall Score`))
if(length(rowsNA) >0){
  metrics_tab <- metrics_tab[-rowsNA, ]
}


# Defining column_info, row_info and palettes
row_info <- data.frame(id = metrics_tab$Method)

column_info <- data.frame(id = colnames(metrics_tab),
                          group = c("Text", "Image", "Text", "Text", "Score overall", 
                                    rep("Removal of batch effects", (1 + n_metrics_batch)),
                                    rep("Cell type label variance", (1 + n_metrics_bio))), 
                          geom = c("text", "image", "text", "text", "bar", "bar", 
                                    rep("circle", n_metrics_batch), "bar", rep("circle", n_metrics_bio)),
                          width = c(8,2.5,2,1.5,2,2, rep(1,n_metrics_batch), 2, rep(1,n_metrics_bio)),
                          overlay = F)

# defining colors palette
palettes <- list("Score overall" = "YlGnBu",
                  "Removal of batch effects" = "BuPu",
                  "Cell type label variance" = "RdPu")



metrics_tab

print('attempting to generate plot at')
g <- scIB_knit_table(data = metrics_tab, column_info = column_info, 
                     row_info = row_info, 
                     palettes = palettes, 
                     usability = F)  

g

ggsave(paste0(outdir, "/summary_metrics.pdf"), g, device = cairo_pdf, width = 297, height = 420, units = "mm")